# [STARTER] Udaplay Project

## Part 01 - Offline RAG

In this part of the project, you'll build your VectorDB using Chroma.

The data is inside folder `project/starter/games`. Each file will become a document in the collection you'll create.
Example.:
```json
{
  "Name": "Gran Turismo",
  "Platform": "PlayStation 1",
  "Genre": "Racing",
  "Publisher": "Sony Computer Entertainment",
  "Description": "A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.",
  "YearOfRelease": 1997
}
```


### Setup

In [11]:
# Only needed for Udacity workspace

import importlib.util
import sys

# Check if 'pysqlite3' is available before importing
if importlib.util.find_spec("pysqlite3") is not None:
    import pysqlite3
    sys.modules['sqlite3'] = sys.modules.pop('pysqlite3')

In [12]:
import os
import json
from pathlib import Path

import chromadb
from chromadb.utils import embedding_functions
from dotenv import load_dotenv

In [13]:
# Create a .env file with the following variables:
# OPENAI_API_KEY="voc-..."              # from Udacity Cloud Resources
# CHROMA_OPENAI_API_KEY="voc-..."       # same Vocareum key for ChromaDB embeddings
# OPENAI_BASE_URL="https://openai.vocareum.com/v1"
# TAVILY_API_KEY="YOUR_KEY"

In [14]:
for env_path in [Path("../../../.env"), Path("../../.env"), Path(".env")]:
    if env_path.exists():
        load_dotenv(env_path)
        break
else:
    load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
CHROMA_OPENAI_API_KEY = os.getenv("CHROMA_OPENAI_API_KEY", OPENAI_API_KEY)
OPENAI_BASE_URL = os.getenv("OPENAI_BASE_URL") or os.getenv(
    "OPENAI_API_BASE", "https://openai.vocareum.com/v1"
)

os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
os.environ["CHROMA_OPENAI_API_KEY"] = CHROMA_OPENAI_API_KEY
os.environ["OPENAI_BASE_URL"] = OPENAI_BASE_URL
os.environ["OPENAI_API_BASE"] = OPENAI_BASE_URL

### VectorDB Instance

In [15]:
COLLECTION_NAME = "udaplay"
CHROMA_PATH = "chromadb"

chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

### Collection

In [16]:
embedding_fn = embedding_functions.OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    api_base=OPENAI_BASE_URL,
)

In [17]:
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn,
)

### Add documents

In [18]:
# Make sure you have a directory "project/starter/games"
data_dir = "games"
ids, documents, metadatas = [], [], []

for file_name in sorted(os.listdir(data_dir)):
    if not file_name.endswith(".json"):
        continue

    file_path = os.path.join(data_dir, file_name)
    with open(file_path, "r", encoding="utf-8") as f:
        game = json.load(f)

    content = (
        f"[{game['Platform']}] {game['Name']} ({game['YearOfRelease']}) - "
        f"Genre: {game['Genre']}. Publisher: {game['Publisher']}. "
        f"{game['Description']}"
    )

    doc_id = os.path.splitext(file_name)[0]
    ids.append(doc_id)
    documents.append(content)
    metadatas.append(game)

collection.upsert(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
)

print(f"Indexed {len(ids)} games into collection '{COLLECTION_NAME}'")

Indexed 15 games into collection 'udaplay'


### Semantic Search Demo

In [19]:
def print_search_results(query: str, n_results: int = 3):
    results = collection.query(query_texts=[query], n_results=n_results, include=["documents", "metadatas", "distances"])
    print(f"Query: {query}\n")

    for rank, (doc, metadata, distance) in enumerate(
        zip(results["documents"][0], results["metadatas"][0], results["distances"][0]),
        start=1,
    ):
        similarity = 1 - distance
        print(f"{rank}. {metadata['Name']} ({metadata['Platform']}, {metadata['YearOfRelease']})")
        print(f"   Similarity: {similarity:.3f}")
        print(f"   {doc}\n")

In [20]:
print_search_results("racing games on PlayStation")
print_search_results("first 3D Mario platformer")
print_search_results("Pokemon games on Game Boy Color")

Query: racing games on PlayStation

1. Gran Turismo 5 (PlayStation 3, 2010)
   Similarity: 0.877
   [PlayStation 3] Gran Turismo 5 (2010) - Genre: Racing. Publisher: Sony Computer Entertainment. A comprehensive racing simulator featuring a vast selection of vehicles and tracks, with realistic driving physics.

2. Gran Turismo (PlayStation 1, 1997)
   Similarity: 0.874
   [PlayStation 1] Gran Turismo (1997) - Genre: Racing. Publisher: Sony Computer Entertainment. A realistic racing simulator featuring a wide array of cars and tracks, setting a new standard for the genre.

3. Mario Kart 8 Deluxe (Nintendo Switch, 2017)
   Similarity: 0.815
   [Nintendo Switch] Mario Kart 8 Deluxe (2017) - Genre: Racing. Publisher: Nintendo. An enhanced version of Mario Kart 8, featuring new characters, tracks, and improved gameplay mechanics.

Query: first 3D Mario platformer

1. Super Mario 64 (Nintendo 64, 1996)
   Similarity: 0.885
   [Nintendo 64] Super Mario 64 (1996) - Genre: Platformer. Publisher: